# SkillLens: extract skills from USAJOBS postings and link them to ESCO (Sprint 3, week 2)

Runs on **Google Colab with a T4 GPU** (*Runtime → Change runtime type → T4 GPU*).

**Before *Run all*, upload two files** with the folder icon 📁 on the left (drag them in):
1. `usajobs_postings.jsonl` from `Documents\SkillLens\data\processed\`
2. `skills_en.csv` from the ESCO download (unzip it first), or the whole ESCO `.zip`

What it does (about 30 minutes):
1. Trains the chosen model on SkillSpan (Skill tagger + Knowledge tagger), same settings as the dev comparison.
2. Tags every sentence of every USAJOBS posting.
3. Links each extracted phrase to its closest ESCO skill using sentence embeddings (cosine similarity).
4. Downloads `usajobs_extracted.jsonl` (put it in `data/processed/`) and `linking_check.csv` (put it in `annotations/`).

In [ ]:
# ---- settings ----
MODEL_NAME = "jjzha/jobspanbert-base-cased"   # best on SkillSpan dev: strict F1 0.554 (bert-base-cased 0.531)
EPOCHS, LR, BATCH, MAX_LEN, SEED = 3, 5e-5, 16, 256, 42
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TAU = 0.60          # minimum cosine similarity to accept an ESCO link (tuned later on linking_check.csv)

import torch, os, glob
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch the runtime to T4 GPU!)")

In [ ]:
!pip -q install transformers sentence-transformers

In [ ]:
# ---- 1. find the uploaded files ----
import json, zipfile, csv, io
if not os.path.exists("usajobs_postings.jsonl"):
    raise SystemExit("Upload usajobs_postings.jsonl first (folder icon on the left).")
postings = [json.loads(l) for l in open("usajobs_postings.jsonl", encoding="utf-8") if l.strip()]
print(len(postings), "postings,", sum(len(p["sentences"]) for p in postings), "sentences")

if not os.path.exists("skills_en.csv"):
    zips = glob.glob("*.zip")
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                if name.endswith("skills_en.csv"):
                    open("skills_en.csv", "wb").write(zf.read(name))
if not os.path.exists("skills_en.csv"):
    raise SystemExit("Upload skills_en.csv (or the ESCO zip) first.")
esco = list(csv.DictReader(open("skills_en.csv", encoding="utf-8")))
print(len(esco), "ESCO skills; columns:", list(esco[0].keys()))

In [ ]:
# ---- 2. SkillSpan training data + span helper (same as the dev notebook) ----
import urllib.request
BASE = "https://raw.githubusercontent.com/kris927b/SkillSpan/main/data/json/"
train = [json.loads(l) for l in urllib.request.urlopen(BASE + "train.json").read().decode("utf-8").splitlines() if l.strip()]
print(len(train), "training sentences")

def bio_to_spans(tags, label):
    spans, start = [], None
    for i, t in enumerate(tags):
        if t.startswith("B") or (t.startswith("I") and start is None):
            if start is not None:
                spans.append((start, i, label))
            start = i
        elif t.startswith("I"):
            continue
        else:
            if start is not None:
                spans.append((start, i, label)); start = None
    if start is not None:
        spans.append((start, len(tags), label))
    return spans

In [ ]:
# ---- 3. train the Skill and Knowledge taggers ----
import random, time, numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
TAG2ID = {"O": 0, "B": 1, "I": 2}; ID2TAG = {v: k for k, v in TAG2ID.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def norm(tag):
    return tag[0] if tag and tag[0] in "BIO" else "O"

def encode(token_lists, tag_lists=None):
    enc = tokenizer(token_lists, is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    if tag_lists is not None:
        labels = []
        for i, tags in enumerate(tag_lists):
            prev, row = None, []
            for w in enc.word_ids(i):
                row.append(-100 if w is None or w == prev else TAG2ID[norm(tags[w])]); prev = w
            labels.append(row)
        enc["labels"] = labels
    return enc

def batches(enc, idx, size):
    for b in range(0, len(idx), size):
        ids = idx[b:b + size]
        feats = [{k: enc[k][i] for k in enc.keys()} for i in ids]
        L = max(len(f["input_ids"]) for f in feats)
        out = {}
        for k in feats[0]:
            pad = -100 if k == "labels" else (tokenizer.pad_token_id if k == "input_ids" else 0)
            out[k] = torch.tensor([f[k] + [pad] * (L - len(f[k])) for f in feats]).to(DEVICE)
        yield ids, out

def train_tagger(column, label):
    enc = encode([s["tokens"] for s in train], [s[column] for s in train])
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label=ID2TAG, label2id=TAG2ID).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    for epoch in range(EPOCHS):
        model.train(); order = list(range(len(train))); random.shuffle(order); total, t0 = 0.0, time.time()
        for _, batch in batches(enc, order, BATCH):
            loss = model(**batch).loss; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(); total += loss.item()
        print(f"[{label}] epoch {epoch + 1}/{EPOCHS} loss {total:.1f} ({time.time() - t0:.0f}s)")
    model.eval()
    return model

def tag(model, token_lists, label):
    enc = encode(token_lists); out = []
    with torch.no_grad():
        for ids, batch in batches(enc, list(range(len(token_lists))), 64):
            logits = model(**batch).logits.argmax(-1).cpu().tolist()
            for j, i in enumerate(ids):
                tags, prev = ["O"] * len(token_lists[i]), None
                for pos, w in enumerate(enc.word_ids(i)):
                    if w is not None and w != prev:
                        tags[w] = ID2TAG[logits[j][pos]]
                    prev = w
                out.append(bio_to_spans(tags, label))
    return out

In [ ]:
# ---- 4. tag every USAJOBS sentence ----
import re
WORD = re.compile(r"[Cc]\+\+|[CcFf]#|\.NET|\w+(?:[-'.&]\w+)*|[^\w\s]")
sent_index = [(pi, s) for pi, p in enumerate(postings) for s in p["sentences"]]
tokens = [WORD.findall(s) for _, s in sent_index]
print(len(tokens), "sentences to tag")

found = [[] for _ in tokens]
for column, label in (("tags_skill", "Skill"), ("tags_knowledge", "Knowledge")):
    model = train_tagger(column, label)
    for k, spans in enumerate(tag(model, tokens, label)):
        found[k].extend(spans)
    del model; torch.cuda.empty_cache()

for p in postings:
    p["spans"] = []
for k, (pi, _) in enumerate(sent_index):
    for start, end, label in found[k]:
        text = " ".join(tokens[k][start:end])
        postings[pi]["spans"].append({"label": label, "text": text})
n = sum(len(p["spans"]) for p in postings)
print(f"{n:,} spans extracted, {n / len(postings):.1f} per posting")

In [ ]:
# ---- 5. link spans to ESCO with sentence embeddings ----
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer(EMBED_MODEL, device=DEVICE)

def clean(t):
    return re.sub(r"\s+", " ", re.sub(r"^[^\w]+|[^\w+#]+$", "", t.lower())).replace(" - ", "-").strip()

# every ESCO concept is searchable by its preferred label and its alternative labels
names, owner = [], []
for i, row in enumerate(esco):
    labels = [row.get("preferredLabel", "")] + (row.get("altLabels") or "").split("\n")
    for lab in {l.strip().lower() for l in labels if l.strip()}:
        names.append(lab); owner.append(i)
print(f"{len(names):,} ESCO labels for {len(esco):,} concepts")
E = emb.encode(names, batch_size=512, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)

unique = sorted({clean(s["text"]) for p in postings for s in p["spans"]} - {""})
Q = emb.encode(unique, batch_size=512, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
best = {}
for b in range(0, len(unique), 2048):
    scores = Q[b:b + 2048] @ E.T
    val, idx = scores.max(dim=1)
    for j, (v, ix) in enumerate(zip(val.tolist(), idx.tolist())):
        best[unique[b + j]] = (owner[ix], v, names[ix])

linked = 0
for p in postings:
    for s in p["spans"]:
        s["norm"] = clean(s["text"])
        if s["norm"] in best:
            ci, score, via = best[s["norm"]]
            row = esco[ci]
            s.update({"esco_uri": row.get("conceptUri", ""), "esco_label": row.get("preferredLabel", ""),
                      "esco_type": row.get("skillType", ""), "matched_label": via, "score": round(score, 4)})
            linked += score >= TAU
total = sum(len(p["spans"]) for p in postings)
print(f"{len(unique):,} unique phrases; {linked:,} of {total:,} spans ({linked / total:.0%}) linked at cosine >= {TAU}")

In [ ]:
# ---- 6. quick look: most frequent ESCO skills (share of postings) ----
from collections import Counter
c = Counter()
for p in postings:
    c.update({s["esco_label"] for s in p["spans"] if s.get("score", 0) >= TAU})
for lab, k in c.most_common(25):
    print(f"{k / len(postings):6.1%}  {lab}")

In [ ]:
# ---- 7. save and download ----
keep = ["usajobs_id", "title", "organization", "job_series", "job_series_name", "grade", "location", "posted", "spans"]
with open("usajobs_extracted.jsonl", "w", encoding="utf-8") as f:
    for p in postings:
        f.write(json.dumps({k: p.get(k) for k in keep}) + "\n")

# 100 random unique phrases for checking the ESCO links by hand (correct = yes / no / partly)
random.seed(SEED)
sample = random.sample(unique, min(100, len(unique)))
with open("linking_check.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["phrase", "esco_label", "matched_label", "score", "correct"])
    for u in sample:
        ci, score, via = best[u]
        w.writerow([u, esco[ci].get("preferredLabel", ""), via, round(score, 3), ""])

meta = {"model": MODEL_NAME, "embed_model": EMBED_MODEL, "tau": TAU, "postings": len(postings),
        "sentences": len(tokens), "spans": total, "unique_phrases": len(unique), "linked_spans": linked,
        "esco_concepts": len(esco)}
json.dump(meta, open("extraction_run.json", "w"), indent=1)
print(meta)
from google.colab import files
for fn in ("usajobs_extracted.jsonl", "linking_check.csv", "extraction_run.json"):
    files.download(fn)